In [ ]:
import os
import pandas as pd
import re
from ollama import Client

top_k = 20
top_p = 0.9
temp = 0.2

batch_size = 20
start_batch = 0      
end_batch = None    

input_path = "all_articles_title_dedup.csv"
output_dir = "LLM_emotion_results"

model_name = "mistral"

os.makedirs(output_dir, exist_ok=True)

# Create Ollama client
client = Client()

def build_prompt(text):
    return f"""
Role of the Model:
You are a financial investor exposed to the energy sector (for example, through equities, infrastructure investments, or venture investments in energy technologies). You are attentive to financial risk, regulatory developments, market opportunities, and long-term sector trends.

Task:
Inhabit this role and infer your internal psychological state upon reading the text.
Do not perform a linguistic analysis of the words used; instead, simulate the gut-level emotion you would feel regarding your capital and market position based on the definition below and assign an intensity score from 0 to 5 for this emotion.
Do not summarise the text.
Do not restate the text.
Do not explain your scores.

Emotion Definitions:
Fear: a basic, intense emotion aroused by the detection of imminent threat, involving an immediate alarm reaction that mobilizes the organism by triggering a set of physiological changes.
Anger: an emotion characterized by tension and hostility arising from frustration, real or imagined injury by another, or perceived injustice.
Disgust: a strong aversion, for example, to the taste, smell, or touch of something deemed revolting, or toward a person or behavior deemed morally repugnant.
Joy: a feeling of extreme gladness, delight, or exultation of the spirit arising from a sense of well-being or satisfaction.
Sadness: an emotional state of unhappiness, ranging in intensity from mild to extreme.
Surprise: an emotion typically resulting from the violation of an expectation or the detection of novelty in the environment.

Emotion Intensity Scale:
0 Absence: absence of emotional reaction with no influence on perception
1 Very weak: barely any emotional reaction that has minimal influence on perception.
2 Weak: A slight emotional reaction is present but remains limited.
3 Moderate: A clear emotional reaction that may influence interpretation of the text.
4 Strong: A strong emotional reaction that could influence judgement or expectations.
5 Very strong: An intense emotional reaction that strongly affects perception and decision-making.

Output Format (ONLY these 6 lines, nothing else):
Fear: <0-5>
Anger: <0-5>
Disgust: <0-5>
Joy: <0-5>
Sadness: <0-5>
Surprise: <0-5>

Text:
{text}
"""
def get_ollama_response(prompt):
    response = client.chat(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
        options={
            'top_k': top_k,
            'top_p': top_p,
            'temperature': temp
        },
        stream=False
    )
    return response['message']['content']


def parse_emotions(response):
    emotions = {
        "Fear": None,
        "Anger": None,
        "Disgust": None,
        "Joy": None,
        "Sadness": None,
        "Surprise": None
    }

    for emotion in emotions.keys():
        match = re.search(rf"\b{emotion}\b\s*:\s*([0-5])", str(response), re.IGNORECASE)
        if match:
            emotions[emotion] = int(match.group(1))

    return emotions

df = pd.read_csv(input_path).reset_index(drop=True)

total_rows = len(df)
total_batches = (total_rows + batch_size - 1) // batch_size

print("Total rows:", total_rows)
print("Batch size:", batch_size)
print("Total batches:", total_batches)

if end_batch is None:
    end_batch = total_batches - 1

if start_batch < 0 or end_batch >= total_batches or start_batch > end_batch:
    raise ValueError("Invalid start_batch or end_batch.")

for batch_num in range(start_batch, end_batch + 1):
    start_row = batch_num * batch_size
    end_row = min(start_row + batch_size - 1, total_rows - 1)

    print("\n" + "=" * 60)
    print(f"Running batch {batch_num} | rows {start_row} to {end_row}")
    print("=" * 60)

    results = []

    for i in range(start_row, end_row + 1):
        print(f"\n===== Article {i} =====")

        text = df.loc[i, "text"]

        if pd.isna(text) or str(text).strip() == "":
            print("Empty text, skip")
            results.append({
                "row_id": i,
                "publication_date": df.loc[i, "publication_date"] if "publication_date" in df.columns else None,
                "document_title": df.loc[i, "document_title"] if "document_title" in df.columns else None,
                "Fear": None,
                "Anger": None,
                "Disgust": None,
                "Joy": None,
                "Sadness": None,
                "Surprise": None,
                "response": None,
                "status": "empty_text"
            })
            continue

        print("Length:", len(str(text)))

        try:
            prompt = build_prompt(str(text))
            response = get_ollama_response(prompt)
            print(f"Article {i}:\n{response}\n")

            parsed = parse_emotions(response)

            results.append({
                "row_id": i,
                "publication_date": df.loc[i, "publication_date"] if "publication_date" in df.columns else None,
                "document_title": df.loc[i, "document_title"] if "document_title" in df.columns else None,
                "Fear": parsed["Fear"],
                "Anger": parsed["Anger"],
                "Disgust": parsed["Disgust"],
                "Joy": parsed["Joy"],
                "Sadness": parsed["Sadness"],
                "Surprise": parsed["Surprise"],
                "response": response,
                "status": "success"
            })

        except Exception as e:
            print(f"Error on article {i}: {e}")
            results.append({
                "row_id": i,
                "publication_date": df.loc[i, "publication_date"] if "publication_date" in df.columns else None,
                "document_title": df.loc[i, "document_title"] if "document_title" in df.columns else None,
                "Fear": None,
                "Anger": None,
                "Disgust": None,
                "Joy": None,
                "Sadness": None,
                "Surprise": None,
                "response": str(e),
                "status": "error"
            })

    # save batch
    results_df = pd.DataFrame(results)
    output_path = os.path.join(
        output_dir,
        f"LLM_emotion_results_batch_{batch_num}_rows_{start_row}_{end_row}.csv"
    )
    results_df.to_csv(output_path, index=False)

    print(f"Saved batch {batch_num} to:")
    print(output_path)
    print(results_df.head())

Total rows: 2194
Batch size: 20
Total batches: 110

Running batch 0 | rows 0 to 19

===== Article 0 =====
Length: 10611
Article 0:
 Fear: 4
Anger: 3
Disgust: 0
Joy: 2
Sadness: 3
Surprise: 5


===== Article 1 =====
Length: 4418
Article 1:
 Fear: 1
Anger: 2
Disgust: 3
Joy: 0
Sadness: 0
Surprise: 4


===== Article 2 =====
Length: 10007
Article 2:
 Fear: 1
Anger: 0
Disgust: 0
Joy: 4
Sadness: 0
Surprise: 3


===== Article 3 =====
Length: 4177
Article 3:
 Fear: 3
Anger: 0
Disgust: 0
Joy: 4
Sadness: 2
Surprise: 4


===== Article 4 =====
Length: 5059
Article 4:
 Fear: 3
Anger: 0
Disgust: 0
Joy: 4
Sadness: 2
Surprise: 4


===== Article 5 =====
Length: 10653
Article 5:
 Fear: 4
Anger: 3
Disgust: 2
Joy: 0
Sadness: 3
Surprise: 4


===== Article 6 =====
Length: 4227
Article 6:
 Fear: 3
Anger: 0
Disgust: 0
Joy: 4
Sadness: 0
Surprise: 4


===== Article 7 =====
Length: 6908
Article 7:
 Fear: 3
Anger: 2
Disgust: 0
Joy: 2
Sadness: 2
Surprise: 4


===== Article 8 =====
Length: 5601
Article 8:
 Fear: 2
An